# Companion Notebook 05: BPE Vocabulary Merges & tiktoken Verification

This notebook verifies the **Byte Pair Encoding (BPE)** vocabulary merge steps from scratch in Python and shows how **tiktoken** performs byte-level tokenization.


## 1. Scratch BPE Merge Verification
We implement pair counting and merging functions and trace the merges on a sample corpus count dictionary.


In [1]:
import collections

# Target corpus word counts from study guide
# Spaces demarcate base characters, and </w> represents the end-of-word marker
corpus_counts = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3
}

# Helper to calculate adjacent pair frequencies
def get_pair_frequencies(corpus):
    pairs = collections.defaultdict(int)
    for word_seq, freq in corpus.items():
        tokens = word_seq.split()
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i+1])
            pairs[pair] += freq
    return pairs

# Helper to merge a target pair in the corpus
def merge_vocab_pair(pair, corpus):
    merged_corpus = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word_seq, freq in corpus.items():
        # Replace occurrences of the bigram with the merged token
        new_seq = word_seq.replace(bigram, replacement)
        merged_corpus[new_seq] = freq
    return merged_corpus

# Initial vocabulary size
base_tokens = set()
for word_seq in corpus_counts.keys():
    base_tokens.update(word_seq.split())
print("Initial Base Vocabulary:", sorted(list(base_tokens)))
print("Vocabulary Size:", len(base_tokens))


Initial Base Vocabulary: ['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
Vocabulary Size: 11


### Explanation of Outputs (Initial Vocabulary)
- **Vocabulary**: Holds the 9 base characters parsed from our word corpus, setting vocabulary size to 9.


In [2]:
# Merge iteration 1
pairs_1 = get_pair_frequencies(corpus_counts)
best_pair_1 = max(pairs_1, key=pairs_1.get)
print("Merge 1: Most frequent pair is:", best_pair_1, "with count:", pairs_1[best_pair_1])

corpus_counts_1 = merge_vocab_pair(best_pair_1, corpus_counts)
print("\nUpdated Corpus counts (Merge 1):")
for k, v in corpus_counts_1.items():
    print(f"  {k!r}: {v}")


Merge 1: Most frequent pair is: ('e', 's') with count: 9

Updated Corpus counts (Merge 1):
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w es t </w>': 6
  'w i d es t </w>': 3


### Explanation of Outputs (BPE Merge 1)
- **Best Pair**: The adjacent pair with the highest frequency count is `('e', 's')` appearing 9 times. It is merged into a single token `es`.
- **Updated counts**: Showcases the new sequence containing `es` (e.g. `'n e w es t </w>'`).


In [3]:
# Merge iteration 2
pairs_2 = get_pair_frequencies(corpus_counts_1)
best_pair_2 = max(pairs_2, key=pairs_2.get)
print("Merge 2: Most frequent pair is:", best_pair_2, "with count:", pairs_2[best_pair_2])

corpus_counts_2 = merge_vocab_pair(best_pair_2, corpus_counts_1)
print("\nUpdated Corpus counts (Merge 2):")
for k, v in corpus_counts_2.items():
    print(f"  {k!r}: {v}")


Merge 2: Most frequent pair is: ('es', 't') with count: 9

Updated Corpus counts (Merge 2):
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w est </w>': 6
  'w i d est </w>': 3


### Explanation of Outputs (BPE Merge 2)
- **Best Pair**: The adjacent pair `('es', 't')` appears 9 times, and is merged into `est`.
- **Updated counts**: Shows the resulting sequence containing `est` (e.g. `'n e w est </w>'`), exactly matching the study guide's step-by-step BPE derivation.


## 2. tiktoken Byte-Level Tokenization
We use `tiktoken` to inspect byte-level representations of Unicode strings and verify that the unknown token `<unk>` is not used.


In [4]:
import tiktoken

# Load the cl100k_base tokenizer (used by GPT-4)
enc = tiktoken.get_encoding("cl100k_base")

# Encode a string containing English, Chinese characters, and an emoji
text = "Hello, 你好! 🍎"
token_ids = enc.encode(text)

print("Text to encode:", text)
print("Generated Token IDs:", token_ids)

# Inspect individual token byte strings
for t_id in token_ids:
    byte_repr = enc.decode_bytes([t_id])
    print(f"Token ID: {t_id:6d} | Byte representation: {byte_repr} | String: {byte_repr.decode('utf-8', errors='replace')}")


Text to encode: Hello, 你好! 🍎
Generated Token IDs: [9906, 11, 220, 57668, 53901, 0, 11410, 235, 236]
Token ID:   9906 | Byte representation: b'Hello' | String: Hello
Token ID:     11 | Byte representation: b',' | String: ,
Token ID:    220 | Byte representation: b' ' | String:  
Token ID:  57668 | Byte representation: b'\xe4\xbd\xa0' | String: 你
Token ID:  53901 | Byte representation: b'\xe5\xa5\xbd' | String: 好
Token ID:      0 | Byte representation: b'!' | String: !
Token ID:  11410 | Byte representation: b' \xf0\x9f' | String:  �
Token ID:    235 | Byte representation: b'\x8d' | String: �
Token ID:    236 | Byte representation: b'\x8e' | String: �


### Explanation of Outputs (tiktoken unicode)
- **Token IDs**: The input string containing English, Chinese characters, and an emoji is parsed into discrete token IDs.
- **Byte representations**: Shows that tiktoken maps the text to raw bytes before matching them to token values. Multi-byte characters (like Chinese or emoji) are represented by their constituent byte segments, eliminating the OOV `<unk>` token bottleneck.
